# Bill Breaker: plain-English, personalized bill explanations

**Author:** Claire Yuan
**Advisor:** Dr. Qingyang Xiao
**Platform:** Google Colab (GPU) | Python | Hugging Face Transformers

### What this notebook does
1. Loads your **100 official congressional bills** (`billsum_100_bills.zip`).
2. Loads the public **BillSum** dataset (about 23,000 US bills, each paired with an official Congressional Research Service summary) and **fine-tunes FLAN-T5** to turn bill text into a plain-English summary. Your 100 bills are held out of training so they can be used as a fair test set.
3. Scores the fine-tuned model against the un-tuned model with ROUGE on your 100 bills.
4. Asks the reader **a few questions** about themselves (state, age, work, household, health coverage, military status, income, interests).
5. Finds the bill sections most relevant to that person and writes a **personalized explanation** with a small instruction-following model.

### Before you run
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Upload `billsum_100_bills.zip` when the upload box appears (or drag it into the Files panel on the left first).

**Important:** These are bills, not laws. Most bills are never enacted, and the BillSum bills date from 1993 to 2018. Model output can be wrong and must be checked against the official text on Congress.gov before anyone relies on it. Nothing here is legal or financial advice.

## 1 · Setup

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "transformers>=4.44", "datasets>=2.20", "accelerate>=0.33",
                       "evaluate", "rouge_score", "sentencepiece"])

import gc, random, re, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
pd.set_option("display.max_colwidth", 120)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU")
if DEVICE == "cpu":
    print("No GPU found. Training on CPU is far too slow: Runtime → Change runtime type → T4 GPU, then rerun.")

Device: cuda | Tesla T4


## 2 · Load your 100 bills

Each `.txt` file is the full text of one bill. The short title is pulled from the standard *"This Act may be cited as ..."* line when present.

In [2]:
BILLS_ZIP = "billsum_100_bills.zip"

if not Path(BILLS_ZIP).exists():
    from google.colab import files
    print("Upload billsum_100_bills.zip")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Upload canceled")
    BILLS_ZIP = next(iter(uploaded))

SHORT_TITLE = re.compile(r"may be cited as the ``(.+?)''", re.S)


def bill_title(text, fallback):
    m = SHORT_TITLE.search(text)
    return " ".join(m.group(1).split()) if m else fallback


rows = []
with zipfile.ZipFile(BILLS_ZIP) as z:
    for name in sorted(z.namelist()):
        if not name.endswith(".txt") or name.startswith("__MACOSX") or "/._" in name:
            continue
        text = z.read(name).decode("utf-8", errors="replace")
        bill_id = Path(name).stem
        rows.append({"bill_id": bill_id, "title": bill_title(text, bill_id), "text": text})

my_bills = pd.DataFrame(rows)
my_bills["words"] = my_bills.text.str.split().str.len()
print(f"{len(my_bills)} bills | words per bill: median {my_bills.words.median():.0f}, "
      f"min {my_bills.words.min()}, max {my_bills.words.max()}")
display(my_bills[["bill_id", "title", "words"]].head(10))

Upload billsum_100_bills.zip


Saving billsum_100_bills.zip to billsum_100_bills.zip
100 bills | words per bill: median 1148, min 632, max 2697


,bill_id,title,words
0,bill_001,bill_001,650
1,bill_002,Human Rights Information Act,2506
2,bill_003,Jackie Robinson Commemorative Coin Act,1541
3,bill_004,bill_004,1257
4,bill_005,Native American Energy Act,2441
5,bill_006,Holocaust Victims Insurance Relief Act of 2001,1232
6,bill_007,bill_007,2234
7,bill_008,Gallatin Land Consolidation Act of 1998,2489
8,bill_009,Marine Debris Act Amendments of 2012,942
9,bill_010,Indian Needs Assessment and Program Evaluation Act of 2001,1282


## 3 · Training data: BillSum (bill text → official plain-language summary)

Your 100 files contain bill text only, with no summaries, so on their own they can't teach a model what a good explanation looks like. BillSum supplies those pairs. Your files look like they come from BillSum, so this cell also looks up each of your bills' official summary (used only for testing) and removes your bills from the training set.

In [3]:
from datasets import load_dataset

billsum = load_dataset("FiscalNote/billsum")
print(billsum)


def norm(text):
    return re.sub(r"\s+", " ", str(text)).strip()


mine = {norm(t): bid for bid, t in zip(my_bills.bill_id, my_bills.text)}
reference = {}
for split_name in billsum:
    for row in billsum[split_name]:
        bid = mine.get(norm(row["text"]))
        if bid:
            reference[bid] = row["summary"]
my_bills["reference_summary"] = my_bills.bill_id.map(reference)
print(f"Found official summaries for {my_bills.reference_summary.notna().sum()} of your {len(my_bills)} bills")

# More training bills = better model but longer training. 2,000 is a reasonable first run on a free T4.
N_TRAIN = 2000
pool = billsum["train"].filter(lambda r: norm(r["text"]) not in mine)
pool = pool.shuffle(seed=SEED).select(range(min(N_TRAIN, len(pool))))
splits = pool.train_test_split(test_size=0.1, seed=SEED)
train_ds, val_ds = splits["train"], splits["test"]
print("Training bills:", len(train_ds), "| validation bills:", len(val_ds))
print("\nExample official summary:\n", train_ds[0]["summary"][:600])

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.8MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 18949
    })
    test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 1237
    })
})
Found official summaries for 100 of your 100 bills


Filter:   0%|          | 0/18949 [00:00<?, ? examples/s]

Training bills: 1800 | validation bills: 200

Example official summary:
 Consumer Energy Commission Act of 2001 - Establishes the Consumer Energy Commission to conduct a nationwide study of significant price spikes in major U.S. consumer energy products during the ten years preceding the date of enactment of this Act, and report its findings to Congress.


## 4 · Fine-tune FLAN-T5 to explain bills

FLAN-T5-base (250M parameters) already follows instructions; fine-tuning teaches it the structure and vocabulary of bill summaries. Bills longer than `MAX_SOURCE` tokens are cut off at the end, which is the main limitation of this model (the median bill here is about 1,600 tokens). T5 is unstable in fp16, so training runs in fp32 with gradient checkpointing to fit in T4 memory.

Training time grows with `N_TRAIN` and `EPOCHS`; lower them for a quick test run.

In [4]:
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)

BASE_MODEL = "google/flan-t5-base"
PROMPT = "Explain this bill in plain English: "
MAX_SOURCE, MAX_TARGET = 1024, 256
EPOCHS = 2

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)


def preprocess(batch):
    enc = tok([PROMPT + t for t in batch["text"]], max_length=MAX_SOURCE, truncation=True)
    enc["labels"] = tok(text_target=batch["summary"], max_length=MAX_TARGET, truncation=True)["input_ids"]
    return enc


cols = train_ds.column_names
train_tok = train_ds.map(preprocess, batched=True, remove_columns=cols)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=cols)

args = Seq2SeqTrainingArguments(
    output_dir="bill-breaker-t5",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    gradient_checkpointing=True,
    logging_steps=25,
    report_to="none",
    seed=SEED,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForSeq2Seq(tok, model=model),
)
trainer.train()
model = trainer.model.eval()
model.config.use_cache = True

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,12.968993,1.472968
2,11.714047,1.452154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


## 5 · Test on your 100 bills (held out from training)

ROUGE measures word overlap with the official summary (higher is better). It compares the fine-tuned model with the original FLAN-T5 on the same bills, so you can see what training bought you. It does not measure factual accuracy; read some outputs yourself too.

In [5]:
import evaluate

N_EVAL = 25  # each bill takes a few seconds to generate; raise for a fuller test


def summarize(text, mdl=None, max_new_tokens=MAX_TARGET):
    mdl = mdl or model
    inputs = tok(PROMPT + text, max_length=MAX_SOURCE, truncation=True, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4,
                           no_repeat_ngram_size=3, early_stopping=True)
    return tok.decode(out[0], skip_special_tokens=True)


test_df = my_bills.dropna(subset=["reference_summary"]).head(N_EVAL).copy()
if test_df.empty:
    print("None of your bills matched BillSum, so there are no reference summaries to score against.")
else:
    rouge = evaluate.load("rouge")
    base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(model.device).eval()
    test_df["base_output"] = [summarize(t, base_model) for t in test_df.text]
    test_df["tuned_output"] = [summarize(t) for t in test_df.text]
    del base_model
    gc.collect()
    torch.cuda.empty_cache()
    scores = pd.DataFrame({
        name: rouge.compute(predictions=test_df[col].tolist(), references=test_df.reference_summary.tolist())
        for name, col in [("FLAN-T5 (no training)", "base_output"), ("Bill Breaker (fine-tuned)", "tuned_output")]
    }).T.round(3)
    display(scores)
    display(test_df[["title", "tuned_output", "reference_summary"]].head(3))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


,rouge1,rouge2,rougeL,rougeLsum
FLAN-T5 (no training),0.209,0.127,0.169,0.180
Bill Breaker (fine-tuned),0.422,0.279,0.341,0.357


,title,tuned_output,reference_summary
0,bill_001,Amends the Internal Revenue Code to prohibit a business entity from being subject to civil liability relating to any...,Shields a business entity from civil liability relating to any injury or death occurring at a facility of that entit...
1,Human Rights Information Act,Human Rights Information Act - Requires the U.S. to continue to receive requests from foreign authorities for legal ...,Human Rights Information Act - Requires certain Federal agencies to identify and organize all human rights records r...
2,Jackie Robinson Commemorative Coin Act,"Jackie Robinson Commemorative Coin Act - Directs the Secretary of the Treasury to mint and issue not more than 500,0...",Jackie Robinson Commemorative Coin Act - Directs the Secretary of the Treasury to: (1) mint and issue one-dollar si...


## 6 · Save the trained model

The model lives inside this Colab session and disappears when it ends. Set `SAVE_TO_DRIVE = True` to keep a copy in your Google Drive and reload it later with `AutoModelForSeq2SeqLM.from_pretrained(path)`.

In [6]:
SAVE_DIR = "bill_breaker_model"
SAVE_TO_DRIVE = True

model.save_pretrained(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    shutil.copytree(SAVE_DIR, "/content/drive/MyDrive/" + SAVE_DIR, dirs_exist_ok=True)
    print("Copied to Google Drive: MyDrive/" + SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to bill_breaker_model


## 7 · Ask the reader a few questions

Every question can be skipped. Answers stay in this session and are only used to choose which parts of the bill to highlight. Set `USE_EXAMPLE_PROFILE = False` to answer the questions yourself.

In [7]:
QUESTIONS = [
    ("state", "Which state do you live in?"),
    ("age", "Age range? (under 18 / 18-24 / 25-44 / 45-64 / 65+)"),
    ("work", "What do you do for work? (job or industry, or student / retired / not working)"),
    ("household", "Who is in your household? (e.g. kids, an elderly parent, just me)"),
    ("health", "How do you get health coverage? (employer / Medicare / Medicaid / marketplace / VA / none)"),
    ("military", "Are you a veteran, service member, or military family? (yes / no)"),
    ("income", "Rough household income? (under $30k / $30-75k / $75-150k / over $150k)"),
    ("interests", "Which issues do you care about most? (e.g. environment, taxes, immigration, schools)"),
]

USE_EXAMPLE_PROFILE = True
EXAMPLE_PROFILE = {
    "state": "Colorado", "age": "18-24", "work": "college student, part-time retail",
    "household": "just me", "health": "on my parents' employer plan", "military": "no",
    "income": "under $30k", "interests": "environment, student loans, public lands",
}


def ask_profile():
    print("Press Enter to skip any question.\n")
    return {key: input(question + " ").strip() for key, question in QUESTIONS}


profile = EXAMPLE_PROFILE if USE_EXAMPLE_PROFILE else ask_profile()
profile = {k: v for k, v in profile.items() if v and v.lower() not in {"no", "none", "skip", "n/a"}}
display(pd.Series(profile, name="answer").to_frame())

,answer
state,Colorado
age,18-24
work,"college student, part-time retail"
household,just me
health,on my parents' employer plan
income,under $30k
interests,"environment, student loans, public lands"


## 8 · Find the parts of the bill that matter to this reader

The bill is split at its `SEC.` headings, then each section is scored by how many profile-related terms it mentions. The term lists are plain keyword rules, so you can read and edit exactly why a section was picked.

In [8]:
SECTION_HEADING = re.compile(
    r"(?im)^\s*(?:SEC(?:TION)?\.?\s+|§\s*)(?P<no>\d+[A-Z0-9]*(?:[-–]\d+[A-Z0-9]*)?)"
    r"\.?\s*(?P<title>[^\n]{0,200})\s*$"
)


def split_bill_sections(bill_text, max_chunk_words=850):
    bill_text = re.sub(r"\r\n?", "\n", str(bill_text or "")).strip()
    matches = list(SECTION_HEADING.finditer(bill_text))
    if matches:
        sections = []
        for i, match in enumerate(matches):
            body = bill_text[match.end(): matches[i+1].start() if i+1 < len(matches) else len(bill_text)].strip()
            sections.append({"section": match.group("no"),
                             "heading": match.group("title").strip(" .-:"), "body": body})
        return sections
    paragraphs = [p.strip() for p in re.split(r"\n{2,}", bill_text) if p.strip()]
    sections, carry, total = [], [], 0
    for p in paragraphs:
        n = len(p.split())
        if carry and total + n > max_chunk_words:
            sections.append({"section": str(len(sections)+1), "heading": "Unnumbered text chunk", "body": "\n\n".join(carry)})
            carry, total = [], 0
        carry.append(p)
        total += n
    if carry:
        sections.append({"section": str(len(sections)+1), "heading": "Unnumbered text chunk", "body": "\n\n".join(carry)})
    return sections


# Answer word -> extra terms to look for in the bill text. Edit freely.
EXPANSIONS = {
    "student": ["student", "college", "university", "tuition", "loan", "education", "school"],
    "retired": ["retire", "Social Security", "pension", "Medicare"],
    "65+": ["Medicare", "Social Security", "elderly", "older", "senior"],
    "45-64": ["retire", "pension"],
    "18-24": ["student", "young", "youth"],
    "under 18": ["minor", "child", "youth", "school"],
    "kids": ["child", "children", "family", "school", "parent"],
    "elderly": ["elderly", "older", "senior", "caregiver", "Medicare"],
    "medicare": ["Medicare"],
    "medicaid": ["Medicaid", "low-income"],
    "marketplace": ["health insurance", "exchange", "premium"],
    "employer": ["employer", "group health plan"],
    "va": ["veteran", "Department of Veterans Affairs"],
    "veteran": ["veteran", "armed forces", "military", "Department of Veterans Affairs"],
    "under $30k": ["low-income", "poverty", "assistance", "credit"],
    "$30-75k": ["tax", "credit", "middle-income"],
    "farm": ["agriculture", "farm", "crop", "rural"],
    "teacher": ["teacher", "school", "education"],
    "nurse": ["nurse", "hospital", "health care"],
    "environment": ["environment", "conservation", "wildlife", "pollution", "climate", "wilderness"],
    "taxes": ["tax", "Internal Revenue Code", "credit", "deduction"],
    "public lands": ["public land", "national forest", "wilderness", "Bureau of Land Management"],
}
MILITARY_TERMS = ["veteran", "armed forces", "military", "service member"]
STOPWORDS = set("just part time with work about most like public".split())


def profile_terms(profile):
    terms = set()
    for key, answer in profile.items():
        low = answer.lower()
        if key in ("work", "interests"):  # free-text answers: use their own words too
            terms.update(w for w in re.findall(r"[a-z][a-z\-]{3,}", low) if w not in STOPWORDS)
        if key == "military" and low.startswith("y"):
            terms.update(MILITARY_TERMS)
        for trigger, extra in EXPANSIONS.items():
            if re.search(r"(?<![a-z])" + re.escape(trigger) + r"(?![a-z])", low):
                terms.update(e.lower() for e in extra)
    if "state" in profile:
        terms.add(profile["state"].lower())
    return sorted(terms)


def rank_sections(text, profile, top_k=3):
    terms = profile_terms(profile)
    rows = []
    for s in split_bill_sections(text):
        body = (s["heading"] + " " + s["body"]).lower()
        hits = {t: len(re.findall(r"\b" + re.escape(t), body)) for t in terms}
        hits = {t: n for t, n in hits.items() if n}
        rows.append({**s, "score": sum(hits.values()), "matched_terms": ", ".join(sorted(hits))})
    ranked = sorted(rows, key=lambda r: -r["score"])
    return [r for r in ranked if r["score"] > 0][:top_k]


print("Terms used for this reader:", ", ".join(profile_terms(profile)))

Terms used for this reader: assistance, bureau of land management, climate, college, colorado, conservation, credit, education, employer, environment, group health plan, lands, loan, loans, low-income, national forest, part-time, pollution, poverty, public land, retail, school, student, tuition, university, wilderness, wildlife, young, youth


## 9 · Write the personalized explanation

The fine-tuned model writes the general summary. A small open instruction model (Qwen2.5-1.5B-Instruct, no API key needed) then rewrites it for this reader, using only the summary and the matched sections, and is told to say so when the bill doesn't clearly affect them. For noticeably better writing you can swap this step for a larger hosted model.

In [9]:
from transformers import pipeline

PERSONALIZER = "Qwen/Qwen2.5-1.5B-Instruct"
writer = pipeline("text-generation", model=PERSONALIZER, torch_dtype=torch.float16, device_map="auto")

SYSTEM = (
    "You explain US congressional bills to ordinary people in plain English at about an 8th-grade reading level. "
    "Use only the bill summary and bill sections you are given; never invent facts, numbers, or effects. "
    "A bill is a proposal and may never have become law, so do not say it is law. "
    "Stay neutral: no political opinions and no advice on how to vote. "
    "Write two short parts with these headings: 'What this bill does' (3 to 5 bullet points) and "
    "'What it could mean for you' (2 to 4 sentences tied to the reader's details). "
    "If the bill does not clearly affect this reader, say that plainly instead of stretching."
)


def explain_bill(text, title, profile, show_sections=True):
    general = summarize(text)
    sections = rank_sections(text, profile)
    excerpts = "\n\n".join(f"SEC. {s['section']} {s['heading']}\n{' '.join(s['body'].split()[:250])}"
                           for s in sections) or "(No sections matched this reader's details.)"
    reader = "\n".join(f"- {k}: {v}" for k, v in profile.items()) or "- (no details given)"
    user = (f"Bill: {title}\n\nGeneral summary:\n{general}\n\n"
            f"Sections most related to the reader:\n{excerpts}\n\nAbout the reader:\n{reader}")
    out = writer([{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
                 max_new_tokens=450, do_sample=False)
    personalized = out[0]["generated_text"][-1]["content"]
    display(Markdown(f"## {title}\n\n**General summary (fine-tuned model):** {general}\n\n---\n\n{personalized}"))
    if show_sections and sections:
        display(pd.DataFrame(sections)[["section", "heading", "score", "matched_terms"]])
    return {"general": general, "personalized": personalized, "sections": sections}

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 10 · Try it

Pick one of your bills by ID (`bill_001` to `bill_100`), or set `INPUT_MODE = "paste"` and paste any bill text from Congress.gov.

In [10]:
INPUT_MODE = "pick"     # "pick" or "paste"
BILL_ID = "bill_009"
PASTED_TITLE = "My bill"
PASTED_TEXT = ""

if INPUT_MODE == "pick":
    row = my_bills.set_index("bill_id").loc[BILL_ID]
    title, text = row.title, row.text
else:
    if len(PASTED_TEXT.strip()) < 80:
        raise ValueError("Paste the bill text into PASTED_TEXT first")
    title, text = PASTED_TITLE, PASTED_TEXT

result = explain_bill(text, title, profile)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


## Marine Debris Act Amendments of 2012

**General summary (fine-tuned model):** Marine Debris Act Amendments of 2012 - Amends the National Oceanic and Atmospheric Administration (NOAA) marine debris research, prevention, and reduction Act to: (1) identify, determine sources of, assess, prevent, reduce, and remove marine debris, with a focus on marine debris posing a threat to living marine resources and navigation safety; (2) provide national and regional coordination to assist States, Indian tribes, and regional organizations in identification, determination of sources, assessment, prevention

---

**What This Bill Does**

This bill aims to improve our understanding and management of marine debris across the country. It will help us:

1. Identify where marine debris comes from.
2. Determine its sources more accurately.
3. Assess the impact of marine debris on our oceans and waterways.
4. Prevent and reduce marine debris.
5. Remove existing marine debris effectively.

By focusing on marine debris that poses a threat to living marine resources and navigation safety, we can better protect our environment and ensure safe travel.

**What It Could Mean For You**

As a college student working part-time at a retail job, your life might be affected indirectly if this bill becomes law. Here’s what you should know:

1. **Environmental Awareness**: The bill encourages awareness about marine debris, which means you'll likely hear more about environmental issues around you.
   
2. **Public Lands**: Since you live near public lands, any new regulations or initiatives aimed at protecting them could influence your daily activities.

3. **Healthcare Costs**: While you're covered under your parents' employer plan, increased funding for marine debris programs might lead to higher healthcare costs down the line due to pollution-related illnesses.

4. **Financial Impact**: Depending on the implementation of new regulations, there might be additional expenses associated with compliance, but overall, the benefits outweigh the costs.

In summary, while you won't directly benefit from this bill, it has broader implications for the environment and public health, potentially affecting your community and lifestyle over time.

,section,heading,score,matched_terms
0,5,NOAA MARINE DEBRIS PROGRAM,6,"education, environment, pollution"
1,2,(33 U.S.C. 1951) is amended to read as follows,1,environment
2,2203,of the Marine Plastic Pollution Research and,1,pollution


## 11 · Limitations and next steps

- **Long bills are cut off** at `MAX_SOURCE` tokens. A next step is summarizing section by section and then combining.
- **ROUGE is not accuracy.** Have people check a sample of outputs against the bill text, and track errors by type (wrong number, wrong agency, invented effect).
- **Personalization is keyword matching plus a small language model.** It can miss indirect effects or overstate weak links. Testing with a set of varied reader profiles and checking each claim against the cited section is the honest way to measure it.
- **Old bills:** BillSum covers 1993–2018. Check status and current text on Congress.gov before showing results to real users.
- **Privacy:** reader answers are personal data. Don't log them, and don't add them to any training set.